# 🏗️ Notebook 1: Distributed Lock Manager — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/distributed-lock-manager
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A **distributed lock**: only one client at a time may hold a named lock across a fleet of
services. Used for things like "only one scheduler runs the nightly job."

### Functional requirements
- `acquire(name, owner, ttl)` → success/failure.
- `release(name, owner)` → releases only if still owner.
- Locks **expire** (TTL) so a dead owner doesn't hold forever.
- Support **renewal** (heartbeat to extend TTL).

### Non-functional
- **Safety**: at most one owner at a time, *even under network partitions and clock skew*.
- **Liveness**: if nobody holds the lock, someone can eventually acquire it.
- **Low latency** (1–5ms local, 10–50ms cross-region).

### Why it's hard
- If owner's process pauses for 60s (GC, VM suspend), its lock may expire; another client
  acquires it. When the first process wakes, both think they hold it.
  → **The classic split-brain.** Solution: **fencing tokens**.


## Architecture options

### 1) Single Redis with `SET NX PX`
```
SET lock:name owner-A NX PX 30000
```
Simple, fast. But if Redis fails, all locks gone. And no fencing token unless you add one.

### 2) Redlock (multi-Redis)
Acquire on majority of M independent Redis instances. Widely debated — Kleppmann's critique
argues it's not safe under arbitrary clock skew.

### 3) Zookeeper / etcd / Consul (consensus-based)
Each relies on Raft/Paxos. A lock = ephemeral znode (ZK) or lease (etcd).
Slower per op, but *correct under partitions*. **Recommended** when correctness matters.

### 4) Database row + TTL
`INSERT INTO locks (name, owner, expires) ... WHERE expires < now()`. Works; DB latency applies.


## Fencing tokens (why they matter)

```
         time ───────────────────────────────────────▶

 Client A: acquire(lock=L, token=17) ────────── GC pause ─── write(X)
                                                              ↑
                                                              still thinks it holds the lock!
 Client B:                  acquire(lock=L, token=18) ── write(X)

 Resource (X):  sees write from A with token=17 **AFTER** a write from B with token=18.
                The resource must **reject** A's stale write because 17 < the last token it saw.
```

The **lock service hands out a monotonically increasing token** with each acquisition.
The protected resource must check `token ≥ last_seen_token` on every operation.
Without this, TTL-based locks *cannot* guarantee mutual exclusion under pauses.
